# 🕌 Athar - Upload to Hugging Face

**Purpose:** Upload Athar datasets to Hugging Face for free, permanent hosting

**Benefits:**
- ✅ Unlimited free storage for public datasets
- ✅ Easy Colab integration
- ✅ Streaming support (no download needed)
- ✅ Version control with git-lfs

**Upload Time:** ~2-4 hours (depends on internet speed)

---

## 1️⃣ Install Hugging Face CLI

In [ ]:
%%bash
pip install -q huggingface_hub
pip install -q git-lfs

echo "✅ Hugging Face CLI installed"

## 2️⃣ Login to Hugging Face

In [ ]:
from huggingface_hub import login, whoami
from google.colab import userdata

print("🔑 Logging in to Hugging Face...")

# Try to get token from Colab secrets
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    if HF_TOKEN:
        login(token=HF_TOKEN)
        user = whoami()
        print(f"✅ Logged in as: {user['name']}")
    else:
        print("⚠️  No HF_TOKEN in secrets")
        print("   Get token from: https://huggingface.co/settings/tokens")
        HF_TOKEN = input("Enter your HF token: ")
        login(token=HF_TOKEN)
        print("✅ Logged in successfully")
except Exception as e:
    print(f"❌ Error: {e}")
    print("\n💡 To save token:")
    print("   1. Go to https://huggingface.co/settings/tokens")
    print("   2. Create new token (write access)")
    print("   3. Add to Colab: Secrets → Add HF_TOKEN")

## 3️⃣ Create Dataset Repository

In [ ]:
from huggingface_hub import HfApi, create_repo

api = HfApi()

# Dataset info
REPO_ID = "Kandil7/Athar-Datasets"  # Change to your username

print(f"📦 Creating dataset repository: {REPO_ID}")

try:
    # Check if repo exists
    api.repo_info(repo_id=REPO_ID, repo_type="dataset")
    print(f"✅ Repository already exists: {REPO_ID}")
except Exception:
    # Create new repository
    create_repo(
        repo_id=REPO_ID,
        repo_type="dataset",
        exist_ok=True
    )
    print(f"✅ Repository created: {REPO_ID}")

print(f"\n🔗 View at: https://huggingface.co/datasets/{REPO_ID}")

## 4️⃣ Prepare Datasets (if not already done)

In [ ]:
import sys
sys.path.insert(0, '/content/athar')

# Run preparation script
!python /content/athar/scripts/prepare_datasets_for_upload.py

## 5️⃣ Upload Files to Hugging Face

In [ ]:
from huggingface_hub import HfApi
from pathlib import Path
import time

api = HfApi()
UPLOAD_DIR = Path("/content/drive/MyDrive/Athar/data/athar-datasets")

print(f"📤 Uploading files from: {UPLOAD_DIR}")
print(f"📦 To repository: {REPO_ID}")

# Get all files to upload
files_to_upload = []
for f in UPLOAD_DIR.rglob('*'):
    if f.is_file():
        rel_path = f.relative_to(UPLOAD_DIR)
        files_to_upload.append((f, str(rel_path)))

print(f"\n📊 Files to upload: {len(files_to_upload)}")
for f, rel in files_to_upload:
    size = f.stat().st_size / 1e9
    print(f"   {rel:40s} {size:.2f} GB")

print(f"\n🚀 Starting upload...")
print(f"   ⏱️  This may take 2-4 hours depending on internet speed")

# Upload each file
for file_path, path_in_repo in files_to_upload:
    try:
        print(f"\n📤 Uploading: {path_in_repo} ({file_path.stat().st_size / 1e9:.2f} GB)")
        
        api.upload_file(
            path_or_fileobj=str(file_path),
            path_in_repo=path_in_repo,
            repo_id=REPO_ID,
            repo_type="dataset"
        )
        
        print(f"   ✅ Uploaded successfully")
        
        # Small delay to avoid rate limiting
        time.sleep(1)
        
    except Exception as e:
        print(f"   ❌ Error: {e}")
        print(f"   ⏭️  Skipping file")

print(f"\n{'='*60}")
print(f"✅ UPLOAD COMPLETE")
print(f"{'='*60}")
print(f"\n🔗 View your dataset at:")
print(f"   https://huggingface.co/datasets/{REPO_ID}")

## 6️⃣ Test Download

In [ ]:
from huggingface_hub import hf_hub_download

print("📥 Testing download...")

# Download a small file first
test_file = "metadata/categories.json"

downloaded = hf_hub_download(
    repo_id=REPO_ID,
    filename=test_file,
    repo_type="dataset"
)

print(f"✅ Downloaded: {downloaded}")

# Read and display
import json
with open(downloaded) as f:
    data = json.load(f)
    print(f"\n📋 Categories: {len(data.get('categories', []))}")

print(f"\n🎉 Dataset is ready for use!")

## 7️⃣ Usage Examples

In [ ]:
# Example 1: Download specific file
from huggingface_hub import hf_hub_download

# Download metadata
metadata = hf_hub_download(
    repo_id=REPO_ID,
    filename="dataset_info.json",
    repo_type="dataset"
)

print(f"✅ Downloaded: {metadata}")

# Example 2: List all files
from huggingface_hub import list_repo_files

files = list_repo_files(REPO_ID, repo_type="dataset")
print(f"\n📁 Files in repository:")
for f in files:
    print(f"   {f}")